# Debugging with Traces and Run Comparisons
Companion notebook for the **Debugging with Traces and Run Comparisons** section of Chapter 9.

The value of a trace is working backward from a bad outcome to the **earliest** bad decision, not just the final failure. Method: start with one bad run, find the first unusual span, compare it with a known-good run for the same task, and check what changed.

Two traces for the same task -- a good run and a bad run that starts identically, then diverges:

In [1]:
good_trace = [
    {"span": "route_request", "tool": None, "duration_ms": 4, "status": "ok", "retries": 0},
    {"span": "policy_lookup", "tool": "policy_api", "duration_ms": 220, "status": "ok", "retries": 0},
    {"span": "refund_lookup", "tool": "payments_api", "duration_ms": 260, "status": "ok", "retries": 0},
    {"span": "draft_reply", "tool": None, "duration_ms": 900, "status": "ok", "retries": 0},
]

bad_trace = [
    {"span": "route_request", "tool": None, "duration_ms": 4, "status": "ok", "retries": 0},
    {"span": "policy_lookup", "tool": "policy_api", "duration_ms": 210, "status": "ok", "retries": 0},
    {"span": "refund_lookup", "tool": "payments_api", "duration_ms": 4800, "status": "timeout", "retries": 3},
    {"span": "refund_lookup_retry", "tool": "payments_api", "duration_ms": 2600, "status": "ok", "retries": 0},
    {"span": "draft_reply", "tool": None, "duration_ms": 910, "status": "ok", "retries": 0},
]

A small comparison utility that finds the **first** span where the two runs diverge, and labels the divergence using the book's own named patterns -- retry storm, state drift, wrong tool choice, or a hidden failure behind normal-looking recovery:

In [2]:
def classify_divergence(good_span: dict, bad_span: dict) -> str:
    if bad_span["retries"] > good_span["retries"] or bad_span["status"] != "ok":
        return "retry_storm_or_tool_error"
    if bad_span.get("tool") != good_span.get("tool"):
        return "wrong_tool_choice"
    if bad_span["duration_ms"] > good_span["duration_ms"] * 2:
        return "latency_spike"
    return "unexplained_difference"


def compare_traces(good: list[dict], bad: list[dict]) -> dict:
    for i, (good_span, bad_span) in enumerate(zip(good, bad)):
        if good_span["span"] != bad_span["span"] or good_span["status"] != bad_span["status"] \
                or bad_span["retries"] != good_span["retries"]:
            return {
                "first_divergent_index": i,
                "span": bad_span["span"],
                "pattern": classify_divergence(good_span, bad_span),
                "good": good_span,
                "bad": bad_span,
            }
    if len(bad) > len(good):
        return {"first_divergent_index": len(good), "span": bad[len(good)]["span"], "pattern": "extra_span_in_bad_run"}
    return {"pattern": "no_divergence_found"}


diagnosis = compare_traces(good_trace, bad_trace)
print(diagnosis)

{'first_divergent_index': 2, 'span': 'refund_lookup', 'pattern': 'retry_storm_or_tool_error', 'good': {'span': 'refund_lookup', 'tool': 'payments_api', 'duration_ms': 260, 'status': 'ok', 'retries': 0}, 'bad': {'span': 'refund_lookup', 'tool': 'payments_api', 'duration_ms': 4800, 'status': 'timeout', 'retries': 3}}


The trace shows the root cause is a **payments_api timeout during `refund_lookup`**, not "the model got worse" -- the final answer (`draft_reply`) looks identical in both runs. Without the trace, you'd only see that this run took 4x longer and never learn why.

Turn the bad trace into a permanent regression case, so the same failure gets caught automatically next time -- ties directly into notebook 03's regression set:

In [3]:
regression_cases = []


def save_as_regression_case(bad_trace: list[dict], diagnosis: dict, task_id: str):
    regression_cases.append({
        "task_id": task_id,
        "trace_snapshot": bad_trace,
        "known_failure_pattern": diagnosis["pattern"],
        "root_cause_span": diagnosis["span"],
    })


save_as_regression_case(bad_trace, diagnosis, task_id="refund_status_check")
print(f"regression set now has {len(regression_cases)} case(s):", [c["task_id"] for c in regression_cases])

regression set now has 1 case(s): ['refund_status_check']
